# 원하는 포즈로 이미지 만들기 — 포즈 조건 이미지 생성 튜토리얼

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
*(GitHub에 올린 뒤 위 링크를 `https://colab.research.google.com/github/<아이디>/pose-image-tool/blob/main/pose_tool.ipynb` 로 바꾸세요)*

**목표**: 사람이 직접 지정한 관절 위치(포즈)와 소품 배치를 조건으로, FLUX.2-klein 4B로 **같은 자세**의 사진을 만들고,
그 결과가 정말 같은 자세인지 **숫자로 검증**합니다.

기준 장면 **BASE-01**: 20대 여성, 검정 요가 상의 + 파란 요가 하의, 거울로 둘러싸인 요가 연습장, 정면 16:9.
화면 왼쪽 벤치 끝에 오른쪽 엉덩이만 걸치고 오른쪽 다리는 벤치 위에서 가부좌처럼 접음, 왼발은 바닥을 디딤,
상체는 화면 오른쪽 가슴 높이의 큰 베개(강철 받침대가 지지)로 크게 기울어 한쪽 뺨을 묻음(옆얼굴 일부 노출), 두 팔로 베개를 감쌈.
따뜻한 노란 조명, 1990년대 필름 질감.

> ⚠️ **중요 — 경로 A는 정식 ControlNet이 아닙니다.**
> FLUX.2-klein 4B에는 공식 OpenPose ControlNet이 없습니다(2026-11 기준, 모델 카드·diffusers 문서 확인).
> 경로 A는 포즈 그림을 Klein의 **참조 이미지(`image=`)** 로 넣고 프롬프트로 "이 자세를 따르라"고 지시하는 방식입니다.
> 자세를 따르는 것이 보장되지 않으므로 모든 결과를 검증하고, 부족하면 **경로 B(SDXL 정식 OpenPose ControlNet → Klein 다듬기)** 로 넘어갑니다.

### 전체 흐름

| 단계 | 셀 | GPU | 하는 일 |
|---|---|---|---|
| 0 | 런타임 확인 | - | GPU/RAM/디스크 확인, 다운로드 없음 |
| 1 | 설치 | - | 라이브러리 설치 |
| 2 | 설정 | - | 폴더·해상도·시드 |
| 3 | **포즈 편집** | - | 관절 좌표·소품 배치를 직접 정의 → 스케치/뼈대/확인용 이미지 저장 → **사람이 확인 후 승인** |
| 4 | 프롬프트 | - | BASE-01 원문과 실험용 변형 |
| 5–6 | 모델 로드 | ✓ | 텍스트 인코더로 임베딩 → 인코더 내림 → 생성 모델 올림 |
| 7 | 검증 도구 | ✓ | 결과에서 OpenPose 재추출 → 목표 뼈대와 비교(PCK, BASE-01 규칙) |
| 8 | 생성 함수 | - | 경로 A / 경로 B |
| 9 | BASE-01 실행 (경로 A) | ✓ | 스케치·뼈대 × 시드 3 → `output_01.png` |
| 10 | 경로 B (필요할 때) | ✓ | SDXL OpenPose ControlNet 초안 → Klein 다듬기 |
| 11 | 실험 경로 결정 | - | BASE-01에서 더 잘 따른 경로 선택 |
| 12 | 실험 2 | ✓ | 같은 포즈, 다른 프롬프트 |
| 13 | 실험 3 | ✓ | 같은 프롬프트, 다른 포즈 |
| 14–15 | 요약·내보내기 | - | 표 정리, zip 다운로드 |
| 16 | **관찰 기록** | - | 무엇을 바꿨고 어떻게 달라졌는지 (실행 후 작성) |

**권장 런타임**: Colab GPU(T4 무료 가능, L4 이상이면 빠름). 3번 포즈 편집 셀들은 GPU 없이도 돌아갑니다.

## 0. 런타임 확인
GPU가 잡혔는지, 메모리와 디스크가 충분한지 먼저 봅니다. 이 셀은 아무것도 내려받지 않습니다.

In [ ]:
# [이 셀이 하는 일] 모델을 받기 전에 실행 환경부터 확인합니다. 아무것도 다운로드하지 않습니다.
# - GPU 종류/VRAM, 시스템 RAM, 남은 디스크를 출력하고 이 노트를 돌릴 수 있는지 판정합니다.
# - 필요 자원(실측 파일 크기 기준):
#     Klein 4B 생성 모델 7.8GB + 텍스트 인코더(Qwen3) 8.0GB + VAE 0.2GB  ≈ 16GB 다운로드
#     검증용 OpenPose body 모델 0.2GB
#     (경로 B만) SDXL base fp16 6.9GB + SDXL OpenPose ControlNet 2.5GB ≈ 9.4GB 추가
# - 이 노트는 텍스트 인코더와 생성 모델을 "따로" 올립니다(5·6번 셀). 그래서 GPU에는 한 번에 최대 약 8~10GB만 올라가
#   무료 Colab T4(15GB)에서도 돌아가도록 짰습니다. L4/A100이면 더 빠릅니다.
import os, shutil, subprocess

def env_report():
    info = {"gpu": None, "vram_gb": 0.0, "ram_gb": None, "disk_free_gb": None}
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=20).stdout.strip()
        if out:
            name, mem = out.splitlines()[0].split(",")
            info["gpu"], info["vram_gb"] = name.strip(), round(int(mem) / 1024, 1)
    except Exception:
        pass
    try:
        import psutil
        info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except Exception:
        pass
    info["disk_free_gb"] = round(shutil.disk_usage(".").free / 1e9, 1)
    return info

ENV = env_report()
print(ENV)
if not ENV["gpu"]:
    print("✗ GPU가 없습니다. Colab 메뉴 [런타임] → [런타임 유형 변경] → GPU(T4 이상)를 고른 뒤 다시 실행하세요.")
    print("  (3번 포즈 편집 셀들은 GPU 없이도 실행됩니다.)")
elif ENV["vram_gb"] < 14:
    print(f"⚠ VRAM {ENV['vram_gb']}GB: Klein 4B 생성 모델(7.8GB)+활성값을 올리기 빠듯합니다. T4(15GB) 이상을 권장합니다.")
else:
    print(f"✓ {ENV['gpu']} {ENV['vram_gb']}GB — 경로 A 실행 가능.",
          "경로 B도 같은 GPU에서 모델을 번갈아 올리는 방식으로 가능합니다." if ENV["vram_gb"] >= 14 else "")
if ENV["disk_free_gb"] is not None and ENV["disk_free_gb"] < 35:
    print(f"⚠ 디스크 여유 {ENV['disk_free_gb']}GB: 경로 A+B 전체(약 26GB) 다운로드에 부족할 수 있습니다.")

## 1. 라이브러리 설치
모델 가중치는 5·6번 셀에서 처음 내려받습니다. 여기서는 코드 라이브러리만 설치합니다.

In [ ]:
# [이 셀이 하는 일] 필요한 라이브러리를 설치합니다 (모델 가중치는 아직 받지 않음).
# - diffusers 0.42 이상: Flux2KleinPipeline(경로 A)과 SDXL ControlNet 파이프라인(경로 B)이 들어 있습니다.
# - transformers 4.51 이상: Klein의 텍스트 인코더(Qwen3)를 읽습니다.
# - controlnet_aux: 생성 결과에서 OpenPose 뼈대를 다시 뽑아 "정말 같은 자세인지" 검증할 때만 씁니다.
#   (주의: OpenPose 전처리기 가중치는 CMU의 비상업 라이선스입니다. 과제·학습 용도로만 사용)
# 설치 후 "런타임 재시작" 안내가 나오면 재시작하고, 이 셀은 건너뛰고 2번 셀부터 다시 실행하세요.
%pip install -q "diffusers>=0.42.0" "transformers>=4.51.0" "accelerate>=1.0" "controlnet_aux==0.0.10" psutil

import diffusers, transformers
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__)
from diffusers import Flux2KleinPipeline   # 여기서 오류가 나면 diffusers가 오래된 것입니다.
print("Flux2KleinPipeline 사용 가능")

## 2. 공통 설정

In [ ]:
# [이 셀이 하는 일] 저장 폴더, 해상도, 시드 등 공통 설정을 한곳에 모읍니다.
# - 결과는 모두 pose-image-tool/samples/ 아래에 저장됩니다. (GitHub에 이 폴더를 그대로 올립니다)
# - Colab은 런타임이 끊기면 파일이 사라집니다. 마지막 셀의 zip 다운로드로 꼭 내려받으세요.
import os, json, gc, time, math
from pathlib import Path

REPO_DIR = Path(os.environ.get("POSE_TOOL_DIR", "pose-image-tool"))
SAMPLES = REPO_DIR / "samples"
DIRS = {
    "base": SAMPLES / "base01_runs",        # BASE-01 경로 A 후보 (시드·참조 종류별)
    "pathB": SAMPLES / "pathB_runs",        # 경로 B 초안(SDXL)과 다듬은 결과(Klein)
    "exp2": SAMPLES / "exp2_same_pose",     # 실험 2: 같은 포즈, 다른 프롬프트
    "exp3": SAMPLES / "exp3_same_prompt",   # 실험 3: 같은 프롬프트, 다른 포즈
}
for d in [SAMPLES, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)

WIDTH, HEIGHT = 1344, 768          # BASE-01 확정 비율 16:9 (포즈 캔버스·SDXL 초안 크기)
# Klein 생성 크기: 출력 이미지와 참조 이미지가 함께 어텐션에 들어가므로 토큰 수가 2배가 됩니다.
# 실측: T4(15GB)에서 1344x768 출력 + 1344x768 참조(토큰 8천여 개)는 어텐션 한 번에 6.6GB를 요구해 CUDA OOM.
# 그래서 VRAM 20GB 미만이면 같은 16:9 비율의 1024x576으로 생성하고, 참조 이미지도 같은 크기로 줄여 넣습니다.
# (포즈 좌표·검증은 비율로 환산하므로 그대로 맞습니다)
GEN_W, GEN_H = (WIDTH, HEIGHT) if globals().get("ENV", {}).get("vram_gb", 0) >= 20 else (1024, 576)
KLEIN_ID = "black-forest-labs/FLUX.2-klein-4B"     # Apache-2.0, 증류 모델(4스텝, guidance 무시)
KLEIN_STEPS = 4
BASE_SEEDS = [101, 202, 303]       # BASE-01 후보 시드
EXP_SEEDS = [101, 202]             # 실험 2·3 시드 (같은 시드로 조건만 바꿔 비교)
RESULTS_JSON = SAMPLES / "results.json"   # 모든 생성·검증 기록이 쌓이는 파일
print("저장 폴더:", SAMPLES.resolve(), "| Klein 생성 크기:", (GEN_W, GEN_H))

## 3. 포즈 편집 — 관절 좌표와 소품 배치를 직접 정합니다

텍스트로 만든 사진에서 바로 뼈대를 뽑으면, **처음 생성된 자세가 틀렸을 때 틀린 자세가 그대로 조건이 됩니다.**
그래서 이 노트는 반대 순서로 갑니다: **사람이 관절 좌표를 먼저 정하고 → 그 좌표로 조건 이미지를 그리고 → 생성 → 검증.**

- 좌표는 1344×768 캔버스의 픽셀 값입니다. 정면 구도이므로 **여성의 오른쪽(`r_*`) = 화면 왼쪽**입니다.
- 고치는 법: 3-1 셀의 숫자를 바꾸고 → 3-1~3-4 셀을 다시 실행 → `samples/pose_0X_layout.png` 확인.
- 3-4에서 출력되는 BASE-01 규칙 검사가 9/9 PASS가 될 때까지 좌표를 고친 뒤, 3-5에서 승인합니다.

### 3-1. 관절 좌표와 소품 상자

In [ ]:
# [이 셀이 하는 일] 포즈를 "사람이 직접 지정한 관절 좌표"로 정의합니다.
# - 텍스트로 만든 이미지에서 뼈대를 뽑으면 처음부터 자세가 틀릴 수 있으므로,
#   BASE-01 자세를 OpenPose 18관절 좌표와 소품 상자로 직접 적어 둡니다.
# - 좌표계: 캔버스 1344x768 픽셀, (0,0)=왼쪽 위. x는 오른쪽, y는 아래로 커집니다.
# - "화면 왼쪽/오른쪽" 기준입니다. 정면 구도이므로 여성의 오른쪽(r_*) = 화면 왼쪽입니다.
# - 값이 None인 관절은 "보이지 않음"(예: 베개에 묻힌 왼쪽 귀)입니다.
# - 수정 방법: 아래 숫자를 바꾸고 이 셀부터 3-4 셀까지 다시 실행 → pose_01_layout.png 로 확인.

CANVAS_W, CANVAS_H = 1344, 768

# OpenPose(COCO-18) 관절 순서. 이 순서를 바꾸면 ControlNet이 읽지 못합니다.
KEYPOINT_NAMES = [
    "nose", "neck",
    "r_shoulder", "r_elbow", "r_wrist",
    "l_shoulder", "l_elbow", "l_wrist",
    "r_hip", "r_knee", "r_ankle",
    "l_hip", "l_knee", "l_ankle",
    "r_eye", "l_eye", "r_ear", "l_ear",
]

# 소품 배치 (BASE-01 확정안: 벤치=화면 왼쪽, 베개=화면 오른쪽 가슴 높이, 강철 받침대가 베개를 지지)
FLOOR_Y = 725
PROPS = {
    # 벤치: 좌석 윗면 y=520, 오른쪽 끝 x=570 (여성의 오른쪽 엉덩이가 이 끝에 걸침)
    "bench_seat": {"box": [140, 520, 570, 548]},
    "bench_legs": {"boxes": [[165, 548, 190, FLOOR_Y], [520, 548, 545, FLOOR_Y]]},
    # 베개: 화면 오른쪽 끝(1344)을 넘어 튀어나온 큰 베개. 위쪽 y≈290, 아래 y≈478 → 가슴(목) 높이
    "pillow": {"box": [895, 290, 1400, 478]},
    # 강철 받침대: 베개 아래 받침판 + 기둥 + 바닥판
    "stand_top": {"box": [980, 478, 1250, 492]},
    "stand_post": {"box": [1100, 492, 1124, FLOOR_Y - 8]},
    "stand_base": {"box": [1020, FLOOR_Y - 8, 1205, FLOOR_Y]},
}

# ---- pose_01: BASE-01 확정 자세 ------------------------------------------------
# 오른쪽 엉덩이만 벤치 끝에 걸침 / 오른쪽 다리는 벤치 위에서 가부좌처럼 접힘 /
# 왼쪽 다리는 바닥을 디딤 / 상체는 화면 오른쪽(여성의 왼쪽)으로 크게 기울어
# 가슴 높이 베개에 한쪽 뺨을 묻음(옆얼굴 일부 노출) / 두 팔로 베개를 끌어안음
POSE_01 = {
    "nose":       (945, 350),
    "neck":       (848, 362),
    "r_shoulder": (820, 300),   # 몸이 기울어 오른쪽 어깨가 위로
    "r_elbow":    (975, 285),   # 오른팔: 베개 윗면을 넘어 감쌈
    "r_wrist":    (1105, 318),
    "l_shoulder": (878, 424),   # 왼쪽 어깨는 아래(베개 쪽)
    "l_elbow":    (990, 470),   # 왼팔: 베개 앞면 아래쪽을 감쌈
    "l_wrist":    (1095, 438),
    "r_hip":      (548, 503),   # 벤치 끝(x=570) 바로 안쪽, 좌석 윗면(y=520) 위
    "r_knee":     (388, 492),   # 벤치 위로 접힌 오른쪽 다리 (무릎이 화면 왼쪽으로)
    "r_ankle":    (505, 518),   # 발목은 사타구니 앞쪽으로 접혀 들어옴 (가부좌)
    "l_hip":      (600, 572),   # 골반이 기울어 왼쪽 엉덩이는 벤치 밖 아래로 떨어짐
    "l_knee":     (760, 610),   # 살짝 굽힌 지지 다리
    "l_ankle":    (840, 716),   # 바닥(y=725)을 디딤, 벤치와 받침대 사이
    "r_eye":      (948, 326),   # 머리가 크게 기울어 두 눈이 위아래로 놓임
    "l_eye":      (968, 358),
    "r_ear":      (915, 312),
    "l_ear":      None,         # 베개에 묻혀 안 보임
}

# ---- pose_02: 실험 3용 — 벤치와 베개 사이에 서서 오른손을 든 단순 자세 ----------
POSE_02 = {
    "nose": (720, 150), "neck": (720, 215),
    "r_shoulder": (665, 226), "r_elbow": (612, 165), "r_wrist": (624, 82),
    "l_shoulder": (775, 226), "l_elbow": (792, 322), "l_wrist": (798, 412),
    "r_hip": (690, 420), "r_knee": (686, 566), "r_ankle": (684, 712),
    "l_hip": (750, 420), "l_knee": (754, 566), "l_ankle": (757, 712),
    "r_eye": (709, 140), "l_eye": (731, 140), "r_ear": (696, 148), "l_ear": (744, 148),
}

# ---- pose_03: 실험 3용 — 벤치에 바르게 앉아 두 손을 무릎에 올린 자세 -------------
POSE_03 = {
    "nose": (360, 238), "neck": (360, 300),
    # 정면에서 본 앉은 자세: 허벅지가 카메라 쪽으로 향해 짧게 보이고(원근 단축), 무릎이 골반보다 넓게 벌어짐
    "r_shoulder": (305, 312), "r_elbow": (284, 438), "r_wrist": (300, 552),
    "l_shoulder": (415, 312), "l_elbow": (436, 438), "l_wrist": (420, 552),
    "r_hip": (335, 512), "r_knee": (298, 562), "r_ankle": (304, 712),
    "l_hip": (385, 512), "l_knee": (422, 562), "l_ankle": (416, 712),
    "r_eye": (349, 228), "l_eye": (371, 228), "r_ear": (336, 236), "l_ear": (384, 236),
}

POSES = {
    "pose_01": {"title": "BASE-01 복합 자세 (엎드림+걸터앉음+한 발 지지)", "kps": POSE_01},
    "pose_02": {"title": "선 자세, 오른손 들기 (실험 3 비교용)", "kps": POSE_02},
    "pose_03": {"title": "벤치에 바르게 앉기 (실험 3 비교용)", "kps": POSE_03},
}

for pid, p in POSES.items():
    missing = [n for n in KEYPOINT_NAMES if n not in p["kps"]]
    assert not missing, f"{pid}: 관절 누락 {missing}"
print("포즈 정의 완료:", ", ".join(f"{k}({v['title']})" for k, v in POSES.items()))

### 3-2. 그리기 함수 (OpenPose 뼈대 / 소품+마네킹 스케치 / 확인용 레이아웃)

In [ ]:
# [이 셀이 하는 일] 관절 좌표를 세 종류 이미지로 그리는 함수를 정의합니다 (GPU 불필요, PIL만 사용).
#   1) render_openpose : 검은 배경 + OpenPose 표준 색 뼈대  → 경로 B(ControlNet) 입력, 경로 A 참조 후보
#   2) render_sketch   : 소품(벤치·베개·강철 받침대) + 굵은 마네킹 → 경로 A(Klein 참조 이미지) 기본 입력
#                        여성의 오른쪽 팔다리=빨강, 왼쪽 팔다리=파랑 (좌우 혼동 방지)
#   3) render_layout   : 2) 위에 격자·좌표·관절 이름·소품 이름을 얹은 "사람 확인용" 이미지 (모델 입력 아님)
from PIL import Image, ImageDraw, ImageFont
import math

def _font(size=14):
    try:
        return ImageFont.load_default(size=size)   # Pillow >= 10.1
    except TypeError:
        return ImageFont.load_default()

# OpenPose 표준 연결(0-based)과 색. controlnet_aux.draw_bodypose 와 같은 규칙입니다.
LIMBS = [(1, 2), (1, 5), (2, 3), (3, 4), (5, 6), (6, 7), (1, 8), (8, 9), (9, 10),
         (1, 11), (11, 12), (12, 13), (1, 0), (0, 14), (14, 16), (0, 15), (15, 17)]
OP_COLORS = [(255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0), (85, 255, 0),
             (0, 255, 0), (0, 255, 85), (0, 255, 170), (0, 255, 255), (0, 170, 255), (0, 85, 255),
             (0, 0, 255), (85, 0, 255), (170, 0, 255), (255, 0, 255), (255, 0, 170), (255, 0, 85)]

def kp_list(kps):
    """{이름: (x,y)} → KEYPOINT_NAMES 순서의 리스트 (없으면 None)."""
    return [kps.get(n) for n in KEYPOINT_NAMES]

def render_openpose(kps, W=None, H=None, stick=4):
    W, H = W or CANVAS_W, H or CANVAS_H
    img = Image.new("RGB", (W, H), (0, 0, 0))
    d = ImageDraw.Draw(img)
    pts = kp_list(kps)
    for (a, b), c in zip(LIMBS, OP_COLORS):
        if pts[a] is None or pts[b] is None:
            continue
        d.line([pts[a], pts[b]], fill=tuple(int(v * 0.6) for v in c), width=stick * 2)
    for p, c in zip(pts, OP_COLORS):
        if p is not None:
            d.ellipse([p[0] - 4, p[1] - 4, p[0] + 4, p[1] + 4], fill=c)
    return img

SKETCH = {
    "bg": (226, 221, 210), "floor": (196, 186, 168), "bench": (140, 92, 46), "bench_dark": (100, 64, 30),
    "pillow": (248, 244, 232), "pillow_edge": (150, 140, 120), "steel": (112, 118, 126), "steel_dark": (70, 75, 82),
    "torso": (40, 40, 44), "right": (214, 48, 48), "left": (38, 88, 212), "head": (232, 196, 166),
}

def draw_props(d, props=None):
    props = props or PROPS
    d.rectangle([0, FLOOR_Y, CANVAS_W, CANVAS_H], fill=SKETCH["floor"])
    for b in props["bench_legs"]["boxes"]:
        d.rectangle(b, fill=SKETCH["bench_dark"])
    d.rectangle(props["bench_seat"]["box"], fill=SKETCH["bench"], outline=SKETCH["bench_dark"], width=3)
    d.rectangle(props["stand_post"]["box"], fill=SKETCH["steel"], outline=SKETCH["steel_dark"], width=2)
    d.rectangle(props["stand_base"]["box"], fill=SKETCH["steel"], outline=SKETCH["steel_dark"], width=2)
    d.rectangle(props["stand_top"]["box"], fill=SKETCH["steel"], outline=SKETCH["steel_dark"], width=2)
    d.rounded_rectangle(props["pillow"]["box"], radius=70, fill=SKETCH["pillow"],
                        outline=SKETCH["pillow_edge"], width=4)

def _head_center(kps):
    pts = [kps[n] for n in ("nose", "r_eye", "l_eye", "r_ear", "l_ear") if kps.get(n)]
    return (sum(p[0] for p in pts) / len(pts), sum(p[1] for p in pts) / len(pts))

def draw_mannequin(d, kps, limb_w=24):
    g = kps.get
    def seg(a, b, color, w=limb_w):
        if g(a) and g(b):
            d.line([g(a), g(b)], fill=color, width=w, joint="curve")
            for p in (g(a), g(b)):
                d.ellipse([p[0] - w / 2, p[1] - w / 2, p[0] + w / 2, p[1] + w / 2], fill=color)
    # 다리 → 몸통 → 팔 → 머리 순 (팔이 몸통 위에 보이도록)
    seg("r_hip", "r_knee", SKETCH["right"]); seg("r_knee", "r_ankle", SKETCH["right"])
    seg("l_hip", "l_knee", SKETCH["left"]);  seg("l_knee", "l_ankle", SKETCH["left"])
    torso = [g("r_shoulder"), g("l_shoulder"), g("l_hip"), g("r_hip")]
    if all(torso):
        d.polygon(torso, fill=SKETCH["torso"])
    seg("neck", "nose", SKETCH["torso"], w=18)
    seg("r_shoulder", "r_elbow", SKETCH["right"]); seg("r_elbow", "r_wrist", SKETCH["right"])
    seg("l_shoulder", "l_elbow", SKETCH["left"]);  seg("l_elbow", "l_wrist", SKETCH["left"])
    cx, cy = _head_center(kps)
    # 머리 크기는 몸통 길이(목~골반 중앙)에 비례 (사람 머리 지름 ≈ 몸통의 0.35~0.4)
    if g("neck") and g("r_hip") and g("l_hip"):
        mh = ((g("r_hip")[0] + g("l_hip")[0]) / 2, (g("r_hip")[1] + g("l_hip")[1]) / 2)
        r = max(28, 0.18 * math.dist(g("neck"), mh))
    else:
        r = 40
    d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=SKETCH["head"], outline=SKETCH["torso"], width=3)
    for n in ("r_eye", "l_eye"):
        if g(n):
            d.ellipse([g(n)[0] - 4, g(n)[1] - 4, g(n)[0] + 4, g(n)[1] + 4], fill=SKETCH["torso"])
    for n in ("r_wrist", "l_wrist", "r_ankle", "l_ankle"):   # 손·발 끝 표시
        if g(n):
            col = SKETCH["right"] if n.startswith("r_") else SKETCH["left"]
            d.ellipse([g(n)[0] - 15, g(n)[1] - 15, g(n)[0] + 15, g(n)[1] + 15], fill=col, outline=(20, 20, 20), width=2)

def render_sketch(kps, props=None, with_props=True):
    img = Image.new("RGB", (CANVAS_W, CANVAS_H), SKETCH["bg"])
    d = ImageDraw.Draw(img)
    if with_props:
        draw_props(d, props)
    draw_mannequin(d, kps)
    return img

def render_layout(kps, title="", props=None, grid=64):
    """사람 확인용: 스케치 + 반투명 OpenPose 뼈대 + 격자/좌표 + 관절·소품 이름."""
    base = render_sketch(kps, props).convert("RGBA")
    over = Image.new("RGBA", base.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(over)
    f, fs = _font(14), _font(12)
    for x in range(0, CANVAS_W + 1, grid):
        d.line([(x, 0), (x, CANVAS_H)], fill=(0, 0, 0, 40), width=1)
        d.text((x + 2, 2), str(x), fill=(0, 0, 0, 160), font=fs)
    for y in range(0, CANVAS_H + 1, grid):
        d.line([(0, y), (CANVAS_W, y)], fill=(0, 0, 0, 40), width=1)
        d.text((2, y + 2), str(y), fill=(0, 0, 0, 160), font=fs)
    op = render_openpose(kps, stick=2).convert("RGBA")
    mask = op.convert("L").point(lambda v: 200 if v > 10 else 0)
    over.paste(op, (0, 0), mask)
    for n in KEYPOINT_NAMES:
        p = kps.get(n)
        if p is None:
            continue
        label = f"{n} ({p[0]},{p[1]})"
        tw = d.textlength(label, font=fs)
        d.rectangle([p[0] + 7, p[1] - 8, p[0] + 11 + tw, p[1] + 8], fill=(255, 255, 255, 190))
        d.text((p[0] + 9, p[1] - 7), label, fill=(0, 0, 0, 255), font=fs)
    props = props or PROPS
    for name, key in (("BENCH", "bench_seat"), ("PILLOW", "pillow"), ("STEEL STAND", "stand_post")):
        b = props[key]["box"]
        d.text((min(b[0] + 8, CANVAS_W - 120), b[1] + 6), name, fill=(0, 0, 0, 230), font=f)
    hidden = [n for n in KEYPOINT_NAMES if kps.get(n) is None]
    legend = (f"{title}   |  red = HER RIGHT (screen left)   blue = HER LEFT (screen right)"
              + (f"   |  hidden: {', '.join(hidden)}" if hidden else ""))
    d.rectangle([0, CANVAS_H - 26, CANVAS_W, CANVAS_H], fill=(255, 255, 255, 215))
    d.text((8, CANVAS_H - 21), legend, fill=(0, 0, 0, 255), font=f)
    return Image.alpha_composite(base, over).convert("RGB")

### 3-3. BASE-01 자세 규칙 (만든 뼈대와 생성 결과를 같은 기준으로 검사)

In [ ]:
# [이 셀이 하는 일] BASE-01 자세 조건을 "검사 가능한 규칙"으로 적어 둡니다.
# - 지금은 직접 만든 뼈대가 기획안과 맞는지 확인하는 데 쓰고,
# - 나중에(8번 셀) 생성 결과에서 다시 뽑은 뼈대가 같은 조건을 지키는지 판정할 때 다시 씁니다.
# - 결과는 PASS / FAIL / N/A(관절을 못 찾아 판정 불가). N/A를 PASS로 세지 않습니다.
# - 생성 이미지 판정 때 소품 위치는 "스케치 배치 그대로 그려졌다"고 가정합니다(허용 오차 포함).
import math

def _angle(a, b, c):
    """b에서의 각도(도). a-b-c."""
    v1 = (a[0] - b[0], a[1] - b[1]); v2 = (c[0] - b[0], c[1] - b[1])
    n = math.hypot(*v1) * math.hypot(*v2)
    return math.degrees(math.acos(max(-1, min(1, (v1[0] * v2[0] + v1[1] * v2[1]) / n)))) if n else float("nan")

def _in_box(p, box, pad=0):
    return box[0] - pad <= p[0] <= box[2] + pad and box[1] - pad <= p[1] <= box[3] + pad

def check_base01_rules(kps, props=None, tol=1.0):
    """tol: 허용 오차 배율 (사람이 만든 뼈대=1.0, 생성 결과 검사=1.5 정도)."""
    props = props or PROPS
    g = kps.get
    seat = props["bench_seat"]["box"]; pillow = props["pillow"]["box"]
    seat_top, bench_r = seat[1], seat[2]
    rules = []
    def rule(name, need, fn):
        if any(g(n) is None for n in need):
            rules.append((name, "N/A", "필요 관절 없음: " + ", ".join(n for n in need if g(n) is None)))
            return
        ok, detail = fn()
        rules.append((name, "PASS" if ok else "FAIL", detail))

    rule("오른쪽 엉덩이만 벤치 끝에 걸침", ["r_hip"], lambda: (
        bench_r - 90 * tol <= g("r_hip")[0] <= bench_r + 15 * tol and seat_top - 50 * tol <= g("r_hip")[1] <= seat_top + 10 * tol,
        f"r_hip={g('r_hip')}, 벤치 끝 x={bench_r}, 좌석 y={seat_top}"))
    rule("왼쪽 엉덩이는 벤치 밖", ["l_hip"], lambda: (
        g("l_hip")[0] > bench_r - 10 * tol, f"l_hip.x={g('l_hip')[0]} > {bench_r}"))
    rule("오른쪽 다리가 벤치 위에서 접힘", ["r_hip", "r_knee", "r_ankle"], lambda: (
        all(abs(g(n)[1] - seat_top) <= 50 * tol and seat[0] - 20 <= g(n)[0] <= bench_r + 20 for n in ("r_knee", "r_ankle"))
        and _angle(g("r_hip"), g("r_knee"), g("r_ankle")) < 70 * tol,
        f"무릎각 {_angle(g('r_hip'), g('r_knee'), g('r_ankle')):.0f}°, r_knee={g('r_knee')}, r_ankle={g('r_ankle')}"))
    rule("왼발이 바닥을 디딤", ["l_ankle", "l_knee", "l_hip"], lambda: (
        g("l_ankle")[1] >= FLOOR_Y - 35 * tol and g("l_ankle")[0] > bench_r,
        f"l_ankle={g('l_ankle')}, 바닥 y={FLOOR_Y}"))
    def lean():
        mh = ((g("r_hip")[0] + g("l_hip")[0]) / 2, (g("r_hip")[1] + g("l_hip")[1]) / 2)
        dx, dy = g("neck")[0] - mh[0], mh[1] - g("neck")[1]
        ang = math.degrees(math.atan2(dx, dy))          # 0°=똑바로 섬, 90°=수평으로 화면 오른쪽
        return ang >= 50 / tol, f"상체 기울기 {ang:.0f}° (수직=0°, 화면 오른쪽 수평=90°)"
    rule("상체가 화면 오른쪽으로 크게 기욺", ["neck", "r_hip", "l_hip"], lean)
    rule("가슴(목)이 베개 높이", ["neck"], lambda: (
        pillow[1] - 40 * tol <= g("neck")[1] <= pillow[3] + 20 * tol, f"neck.y={g('neck')[1]}, 베개 y={pillow[1]}~{pillow[3]}"))
    rule("얼굴이 베개에 닿음", ["nose"], lambda: (
        _in_box(g("nose"), pillow, pad=45 * tol), f"nose={g('nose')}, 베개 x≥{pillow[0]}"))
    rule("두 팔이 베개를 감쌈", ["r_wrist", "l_wrist"], lambda: (
        _in_box(g("r_wrist"), pillow, 30 * tol) and _in_box(g("l_wrist"), pillow, 30 * tol),
        f"r_wrist={g('r_wrist')}, l_wrist={g('l_wrist')}"))
    rule("옆얼굴 일부 노출(코+눈 하나 이상)", ["nose"], lambda: (
        g("r_eye") is not None or g("l_eye") is not None, f"r_eye={g('r_eye')}, l_eye={g('l_eye')}"))
    return rules

def print_rules(rules, title=""):
    n_pass = sum(r[1] == "PASS" for r in rules)
    print(f"[{title}] 규칙 통과 {n_pass}/{len(rules)}  (N/A는 통과로 세지 않음)")
    for name, st, detail in rules:
        mark = {"PASS": "O", "FAIL": "X", "N/A": "?"}[st]
        print(f"  {mark} {st:4s} | {name:22s} | {detail}")
    return n_pass, len(rules)

### 3-4. 중간 결과물 저장과 확인
저장된 `pose_01_layout.png`에서 **뼈대, 벤치, 베개, 강철 받침대 위치**를 눈으로 확인하세요.

In [ ]:
# [이 셀이 하는 일] 세 포즈의 중간 결과물을 samples/ 에 저장하고 화면에 띄웁니다.
#   pose_0X.png           : 모델에 넣는 포즈 참조 이미지 (소품 + 마네킹 스케치) ← 경로 A 기본 입력
#   pose_0X_openpose.png  : OpenPose 형식 뼈대 (검은 배경)                     ← 경로 B 입력, 경로 A 비교용
#   pose_0X_layout.png    : 사람 확인용 (격자·좌표·관절/소품 이름)              ← 이걸 보고 위치를 확인
#   pose_0X_keypoints.json: 관절 좌표 + 소품 상자 (다시 불러오거나 수정할 때 사용)
# 마지막에 pose_01이 BASE-01 규칙을 모두 지키는지 출력합니다. 하나라도 FAIL이면 3-1 셀 좌표를 고치세요.
import json

def save_pose_assets(pid):
    p = POSES[pid]
    kps = p["kps"]
    render_sketch(kps).save(SAMPLES / f"{pid}.png")
    render_openpose(kps).save(SAMPLES / f"{pid}_openpose.png")
    render_layout(kps, title=f"{pid}: {pid.upper()}").save(SAMPLES / f"{pid}_layout.png")
    with open(SAMPLES / f"{pid}_keypoints.json", "w", encoding="utf-8") as f:
        json.dump({"pose_id": pid, "title": p["title"], "canvas": [CANVAS_W, CANVAS_H],
                   "keypoint_order": KEYPOINT_NAMES,
                   "keypoints": {n: (list(kps[n]) if kps.get(n) else None) for n in KEYPOINT_NAMES},
                   "props": PROPS, "floor_y": FLOOR_Y}, f, ensure_ascii=False, indent=2)

for pid in POSES:
    save_pose_assets(pid)
print("저장 위치:", SAMPLES)
print(sorted(x.name for x in SAMPLES.glob("pose_0*")))

TARGET_RULES = check_base01_rules(POSES["pose_01"]["kps"])
n_ok, n_all = print_rules(TARGET_RULES, "pose_01 수동 뼈대 vs BASE-01 기획안")
if n_ok != n_all:
    print("\n⚠ 수동 뼈대가 기획안 규칙을 다 지키지 못합니다. 생성 전에 좌표를 고치세요.")

try:
    from IPython.display import display
    for pid in POSES:
        print(pid, "-", POSES[pid]["title"])
        display(Image.open(SAMPLES / f"{pid}_layout.png").resize((CANVAS_W // 2, CANVAS_H // 2)))
except ImportError:
    pass

### 3-5. 포즈 승인
확인이 끝났으면 아래 값을 `True`로 바꾸고 실행합니다. `False`이면 생성 셀이 멈춥니다.

In [ ]:
# [이 셀이 하는 일] 사람이 포즈를 확인했다는 표시입니다.
# samples/pose_01_layout.png 에서 뼈대·벤치·베개·강철 받침대 위치를 확인한 뒤에만 True로 바꾸세요.
# False인 동안에는 생성 셀이 멈춥니다 (틀린 포즈로 GPU 시간을 쓰지 않기 위해).
POSE_APPROVED = True   # v1 승인 (2026-11-13): 완벽하지 않지만 전체 자세 구조가 전달된다고 판단

## 4. 프롬프트
BASE-01 원문은 고정입니다. 실험 2의 변형 프롬프트는 "프롬프트만 바꾸는 실험"용입니다. 전체 목록은 `prompts.md`에도 있습니다.

In [ ]:
# [이 셀이 하는 일] 실험에 쓰는 프롬프트를 모두 정의합니다. (prompts.md 와 같은 내용)
# - BASE_01 : 확정된 기준 프롬프트 원문. 고치지 않습니다.
# - 실험 2의 변형(P2_*)은 "프롬프트만 바꾸는 실험"을 위한 것이고, BASE-01 자체를 바꾸는 것이 아닙니다.
# - "klein" 은 Klein 4B용(긴 자연어), "sdxl" 은 경로 B 초안용입니다.
#   SDXL의 CLIP은 77토큰까지만 읽으므로 짧게 줄인 판이고, 세부 묘사는 Klein이 다듬는 단계에서 넣습니다.

BASE_01 = (
    "A 1990s retro film photograph, YouTube-thumbnail style eye-catching composition, 16:9 wide shot, front view, "
    "full body visible from head to feet. A woman in her twenties wearing a black yoga top and blue yoga leggings is "
    "in the center of a yoga studio surrounded by mirrors on all walls. She holds a deliberately bizarre, unstable "
    "pose. On the left side of the frame stands a wooden bench; only her right hip is perched on the very edge of the "
    "bench, and her right leg is folded cross-legged on top of the bench. Her left leg is planted firmly on the floor "
    "on the right side of the frame, slightly bent, supporting her weight. Her upper body leans strongly toward the "
    "right side of the frame, forming a dramatic diagonal line, and she hugs a large, fluffy pillow that sticks out "
    "from the right edge of the frame at chest height, held up by a sturdy steel stand underneath. One cheek is "
    "pressed into the pillow, with half of her face still visible in profile. Both arms wrap around the pillow, with "
    "elbows and wrists clearly visible and not hidden behind her torso. She looks like she is lying face-down, "
    "sitting, and standing on one leg all at once. Warm yellow lighting, soft film grain, slightly faded colors, "
    "gentle vintage softness, natural skin texture. No text, no logos."
)

# 자세 문장만 뺀 판 (인물·복장·소품·배치·조명·질감 문장은 BASE_01과 동일) → 실험 3에서 "포즈 조건만" 비교할 때 사용
BASE_01_NEUTRAL = (
    "A 1990s retro film photograph, YouTube-thumbnail style eye-catching composition, 16:9 wide shot, front view, "
    "full body visible from head to feet. A woman in her twenties wearing a black yoga top and blue yoga leggings is "
    "in the center of a yoga studio surrounded by mirrors on all walls. On the left side of the frame stands a wooden "
    "bench. A large, fluffy pillow sticks out from the right edge of the frame at chest height, held up by a sturdy "
    "steel stand underneath. Warm yellow lighting, soft film grain, slightly faded colors, gentle vintage softness, "
    "natural skin texture. No text, no logos."
)

SDXL_BASE = ("1990s film photo, young woman in black yoga top and blue yoga leggings, mirrored yoga studio, "
             "wooden bench on left, big fluffy pillow on steel stand on right, warm yellow light, film grain, "
             "full body, front view")
SDXL_NEG = ("text, logo, watermark, extra limbs, extra legs, extra arms, missing limbs, deformed hands, "
            "multiple people, cropped body, cartoon, drawing, stick figure")

PROMPTS = {
    "BASE_01": {"desc": "확정 기준 프롬프트 (대조군)", "klein": BASE_01, "sdxl": SDXL_BASE},
    "BASE_01_NEUTRAL": {"desc": "BASE_01에서 자세 문장만 제거", "klein": BASE_01_NEUTRAL, "sdxl": SDXL_BASE},
    "P2_1_SUBJECT": {
        "desc": "인물·복장만 변경 (50대 남성, 회색 정장)",
        "klein": BASE_01.replace("A woman in her twenties wearing a black yoga top and blue yoga leggings",
                                 "A man in his fifties wearing a gray business suit and a white shirt")
                        .replace(" her ", " his ").replace("She ", "He ").replace(" she ", " he ")
                        .replace("her right", "his right").replace("her left", "his left")
                        .replace("Her ", "His ").replace("of her face", "of his face"),
        "sdxl": SDXL_BASE.replace("young woman in black yoga top and blue yoga leggings",
                                  "middle-aged man in gray business suit"),
    },
    "P2_2_STYLE": {
        "desc": "질감·조명만 변경 (현대 디지털 사진, 차가운 백색 주광)",
        "klein": BASE_01.replace("A 1990s retro film photograph", "A sharp modern digital photograph")
                        .replace("Warm yellow lighting, soft film grain, slightly faded colors, gentle vintage softness",
                                 "Cool white daylight, crisp details, vivid clean colors, high dynamic range"),
        "sdxl": SDXL_BASE.replace("1990s film photo", "modern digital photo")
                         .replace("warm yellow light, film grain", "cool white daylight, sharp"),
    },
    "P2_3_CONFLICT": {
        "desc": "자세 문장을 '똑바로 선 자세'로 바꿔 포즈 조건과 충돌시킴",
        "klein": BASE_01_NEUTRAL.replace(
            "On the left side of the frame stands a wooden bench.",
            "She stands upright between the bench and the pillow with both feet on the floor and her arms relaxed "
            "at her sides. On the left side of the frame stands a wooden bench."),
        "sdxl": SDXL_BASE + ", standing upright, arms at sides",
    },
}

# 경로별 지시문. 같은 장면 프롬프트 앞에 붙입니다.
PREFIX = {
    # 경로 A-스케치: 소품+마네킹 스케치(pose_0X.png)를 참조 이미지로 넣을 때
    "A_sketch": (
        "Image 1 is a flat color pose-and-layout sketch, not a style reference. Copy only the body pose and the "
        "positions of the objects from it. In the sketch the black block is the torso, the red limbs are the person's "
        "right arm and right leg, the blue limbs are the left arm and left leg, the brown table is the bench, the "
        "white rounded block is the pillow and the gray post is the steel stand. Generate a new, fully photographic "
        "image in which the person holds exactly this pose and the objects are in the same places. Do not draw the "
        "sketch, colored limbs, lines or dots. Scene: "),
    # 경로 A-뼈대: OpenPose 뼈대(pose_0X_openpose.png)를 참조 이미지로 넣을 때
    "A_openpose": (
        "Image 1 is an OpenPose skeleton: colored lines and dots on black marking the joints of one person (head, "
        "shoulders, elbows, wrists, hips, knees, ankles). It is a pose guide only. Generate a new, fully photographic "
        "image in which the person's joints are at the same positions as in the skeleton. Do not draw the skeleton, "
        "lines, dots or the black background. Scene: "),
    # 경로 B-다듬기: SDXL ControlNet 초안을 Klein에 넣어 질감·인물을 다듬을 때
    "B_refine": (
        "Image 1 is a draft photo. Keep the person's exact body pose, limb positions, head position, camera framing "
        "and the positions of the bench, the pillow and the steel stand unchanged. Only improve realism, anatomy, "
        "clothing, lighting and texture to match this description: "),
}

def klein_text(prompt_key, mode):
    return PREFIX[mode] + PROMPTS[prompt_key]["klein"]

with open(SAMPLES / "prompts_used.json", "w", encoding="utf-8") as f:
    json.dump({"PROMPTS": PROMPTS, "PREFIX": PREFIX, "SDXL_NEG": SDXL_NEG}, f, ensure_ascii=False, indent=2)
for k, v in PROMPTS.items():
    print(f"{k:16s} | {v['desc']}")

## 5. 모델 로드 ① — 텍스트 인코더로 프롬프트 임베딩 만들기
처음 실행할 때 Klein 4B 텍스트 인코더(약 8GB)를 내려받습니다.

In [ ]:
# [이 셀이 하는 일] 모델 로드 1단계 — Klein 4B의 텍스트 인코더(Qwen3, 8GB)만 올려 모든 프롬프트를 미리 숫자(임베딩)로 바꿉니다.
# - 왜 나누나: 텍스트 인코더(8GB)+생성 모델(7.8GB)을 한꺼번에 올리면 무료 Colab(T4 15GB, RAM 12.7GB)에서 메모리가 넘칩니다.
#   임베딩만 만들어 두고 인코더는 지우면, 다음 셀에서 생성 모델만 올리면 됩니다.
# - 프롬프트(4번 셀)를 고쳤다면 이 셀을 다시 실행해야 바뀐 문장이 반영됩니다.
import torch
from diffusers import Flux2KleinPipeline

assert ENV["gpu"], "GPU 런타임이 필요합니다 (0번 셀 참고)."
DEVICE, DTYPE = "cuda", torch.bfloat16

def encode_all_prompts():
    t0 = time.time()
    te_pipe = Flux2KleinPipeline.from_pretrained(KLEIN_ID, transformer=None, vae=None, torch_dtype=DTYPE)
    te_pipe.to(DEVICE)
    embeds = {}
    with torch.no_grad():
        for key in PROMPTS:
            for mode in PREFIX:
                pe, _ = te_pipe.encode_prompt(prompt=klein_text(key, mode), device=DEVICE)
                embeds[(key, mode)] = pe.to("cpu")
    del te_pipe
    gc.collect(); torch.cuda.empty_cache()
    print(f"임베딩 {len(embeds)}개 완료 ({time.time() - t0:.0f}s). 텍스트 인코더는 메모리에서 내렸습니다.")
    return embeds

EMBEDS = encode_all_prompts()

## 6. 모델 로드 ② — Klein 4B 생성 모델 올리기
텍스트 인코더 없이 생성 모델(약 7.8GB)+VAE만 GPU에 올립니다. 경로 B용 SDXL 함수도 여기서 정의하지만, 실제로 받는 것은 10번 셀에서 필요할 때뿐입니다.

In [ ]:
# [이 셀이 하는 일] 모델 로드 2단계 — 생성 모델을 올리고 내리는 함수를 정의하고, Klein 4B 생성 모델을 올립니다.
# - ensure_klein(): Klein 4B 생성 모델(텍스트 인코더 없이)을 GPU에 올림. SDXL이 올라가 있으면 먼저 내림.
# - ensure_sdxl(): 경로 B용 SDXL + OpenPose ControlNet을 올림. Klein이 올라가 있으면 먼저 내림.
# - 두 모델을 동시에 올리지 않으므로 GPU에는 한 번에 한 모델만 있습니다.
MODELS = {"klein": None, "sdxl": None}

def unload(name):
    if MODELS.get(name) is not None:
        MODELS[name] = None
        gc.collect(); torch.cuda.empty_cache()
        print(f"[{name}] 메모리에서 내림")

def ensure_klein():
    if MODELS["klein"] is None:
        unload("sdxl")
        t0 = time.time()
        pipe = Flux2KleinPipeline.from_pretrained(KLEIN_ID, text_encoder=None, tokenizer=None, torch_dtype=DTYPE)
        pipe.to(DEVICE)
        pipe.set_progress_bar_config(disable=True)
        MODELS["klein"] = pipe
        print(f"[klein] 로드 완료 {time.time() - t0:.0f}s | GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB")
    return MODELS["klein"]

# 경로 B 모델 (라이선스·크기는 10번 셀 설명 참고)
SDXL_ID = "stabilityai/stable-diffusion-xl-base-1.0"          # CreativeML Open RAIL++-M, fp16 변형 6.9GB
SDXL_POSE_CN_ID = "xinsir/controlnet-openpose-sdxl-1.0"         # Apache-2.0, 2.5GB, SDXL용 ControlNetModel

def ensure_sdxl():
    if MODELS["sdxl"] is None:
        unload("klein")
        from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
        t0 = time.time()
        cn = ControlNetModel.from_pretrained(SDXL_POSE_CN_ID, torch_dtype=torch.float16)
        pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
            SDXL_ID, controlnet=cn, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
        pipe.to(DEVICE)
        pipe.set_progress_bar_config(disable=True)
        MODELS["sdxl"] = pipe
        print(f"[sdxl] 로드 완료 {time.time() - t0:.0f}s | GPU 사용 {torch.cuda.memory_allocated() / 1e9:.1f}GB")
    return MODELS["sdxl"]

ensure_klein();

## 7. 검증 도구 — 생성 결과가 정말 같은 자세인가?
결과 사진에서 OpenPose 뼈대를 다시 뽑아 목표 좌표와 비교합니다. 초록=목표, 빨강=검출. 숫자는 보조 지표이고, 최종 판단은 `_verify.png`를 눈으로 봅니다.

In [ ]:
# [이 셀이 하는 일] 결과 검증 도구 — 생성된 사진에서 OpenPose 뼈대를 "다시" 뽑아 목표 뼈대와 비교합니다.
# - 눈으로만 "비슷해 보인다"고 하지 않고 숫자로 남기기 위한 장치입니다.
# - PCK: 목표 관절에서 (몸통 길이 × 0.2) 안쪽에 검출된 관절의 비율. 관절을 못 찾으면 틀린 것으로 셉니다.
# - 판정: SUCCESS(PCK≥0.8, pose_01은 BASE-01 규칙도 모두 PASS) / PARTIAL(PCK≥0.5) / FAIL / NO_DETECTION(사람을 못 찾음)
#   (이미지 위 글자는 한글 폰트가 없어도 보이도록 영어로 씁니다)
# - 함께 표시하는 경고: 좌우 반전 의심, 사람 여러 명(거울 반사), 참조 스케치·뼈대가 그림에 그대로 남음
# - 한계: OpenPose 자체가 이렇게 기이한 자세를 잘못 읽을 수 있습니다. 최종 판단은 겹쳐 그린 _verify.png를 눈으로 봅니다.
import numpy as np
from huggingface_hub import hf_hub_download
from controlnet_aux.open_pose import OpenposeDetector
from controlnet_aux.open_pose.body import Body

# body 모델(0.2GB)만 받습니다. (from_pretrained는 손·얼굴 모델까지 0.3GB를 더 받으므로 쓰지 않음)
_body = Body(hf_hub_download("lllyasviel/Annotators", "body_pose_model.pth")).to(DEVICE)
POSE_DETECTOR = OpenposeDetector(_body)

EVAL_JOINTS = ["nose", "neck", "r_shoulder", "r_elbow", "r_wrist", "l_shoulder", "l_elbow", "l_wrist",
               "r_hip", "r_knee", "r_ankle", "l_hip", "l_knee", "l_ankle"]   # 눈·귀는 작아서 평가에서 뺌
SWAP = {n: (n.replace("r_", "l_") if n.startswith("r_") else n.replace("l_", "r_") if n.startswith("l_") else n)
        for n in KEYPOINT_NAMES}

def detect_people(img):
    """이미지 → [{관절이름: (x, y) or None}, ...] (사람 수만큼, 포즈 캔버스 1344x768 기준 좌표로 환산)"""
    W, H = CANVAS_W, CANVAS_H     # 검출 좌표는 0~1 비율이므로 생성 크기와 상관없이 캔버스 좌표로 바꿉니다
    poses = POSE_DETECTOR.detect_poses(np.array(img.convert("RGB")))
    people = []
    for p in poses:
        kps = {}
        for name, k in zip(KEYPOINT_NAMES, p.body.keypoints):
            kps[name] = None if k is None else (float(k.x * W), float(k.y * H))
        people.append(kps)
    return people

def _torso_len(kps):
    mh = ((kps["r_hip"][0] + kps["l_hip"][0]) / 2, (kps["r_hip"][1] + kps["l_hip"][1]) / 2)
    return math.dist(kps["neck"], mh)

def pck(target, det, thr=0.2):
    lim = thr * _torso_len(target)
    errs = {}
    for n in EVAL_JOINTS:
        if target.get(n) is None:
            continue
        errs[n] = None if det.get(n) is None else math.dist(target[n], det[n])
    hits = sum(1 for e in errs.values() if e is not None and e <= lim)
    return hits / len(errs), errs, lim

def leak_score(img, ref_kind):
    """참조 이미지가 결과에 그대로 찍혔는지 대략 추정 (스케치의 빨강/파랑 팔다리, 뼈대의 검은 배경 비율)."""
    a = np.asarray(img.convert("RGB")).astype(int)
    if ref_kind == "sketch":
        red = (np.abs(a - [214, 48, 48]).sum(-1) < 60).mean()
        blue = (np.abs(a - [38, 88, 212]).sum(-1) < 60).mean()
        return float(red + blue)
    return float((a.sum(-1) < 30).mean())

def verify(img, pose_id, ref_kind=None):
    target = POSES[pose_id]["kps"]
    people = detect_people(img)
    rec = {"n_people": len(people), "pck": 0.0, "pck_swapped": 0.0, "mean_err_px": None,
           "rules_pass": None, "rules_total": None, "flags": [], "verdict": "NO_DETECTION", "detected": None, "per_joint": {}}
    if people:
        scored = [(pck(target, p)[0], p) for p in people]
        best_score, best = max(scored, key=lambda t: t[0])
        score, errs, lim = pck(target, best)
        swapped = {n: best.get(SWAP[n]) for n in KEYPOINT_NAMES}
        rec.update(pck=round(score, 3), pck_swapped=round(pck(target, swapped)[0], 3), detected=best,
                   per_joint={n: (None if e is None else round(e, 1)) for n, e in errs.items()},
                   pck_threshold_px=round(lim, 1))
        found = [e for e in errs.values() if e is not None]
        rec["mean_err_px"] = round(sum(found) / len(found), 1) if found else None
        missing = [n for n, e in errs.items() if e is None]
        if missing:
            rec["flags"].append("missing joints: " + ", ".join(missing))
        if rec["pck_swapped"] > rec["pck"] + 0.15:
            rec["flags"].append("left/right swapped?")
        if pose_id == "pose_01":
            rules = check_base01_rules(best, tol=1.5)
            rec["rules"] = rules
            rec["rules_pass"] = sum(r[1] == "PASS" for r in rules)
            rec["rules_total"] = len(rules)
        rules_ok = rec["rules_pass"] is None or rec["rules_pass"] == rec["rules_total"]
        rec["verdict"] = ("SUCCESS" if score >= 0.8 and rules_ok else "PARTIAL" if score >= 0.5 else "FAIL")
    if len(people) > 1:
        rec["flags"].append(f"{len(people)} people detected (mirror reflection?)")
    if ref_kind:
        ls = leak_score(img, ref_kind)
        rec["leak"] = round(ls, 3)
        if (ref_kind == "sketch" and ls > 0.02) or (ref_kind == "openpose" and ls > 0.35):
            rec["flags"].append("reference image may be copied into output")
    return rec

def draw_verify(img, pose_id, rec, label=""):
    """결과 위에 목표 뼈대(초록)와 검출 뼈대(빨강)를 겹쳐 그린 확인용 이미지."""
    out = img.convert("RGB").resize((CANVAS_W, CANVAS_H))   # 목표 뼈대(캔버스 좌표)와 겹치기 위해 캔버스 크기로
    d = ImageDraw.Draw(out)
    def skel(kps, color, w):
        pts = kp_list(kps)
        for a, b in LIMBS:
            if pts[a] and pts[b]:
                d.line([pts[a], pts[b]], fill=color, width=w)
        for p in pts:
            if p:
                d.ellipse([p[0] - 5, p[1] - 5, p[0] + 5, p[1] + 5], fill=color)
    skel(POSES[pose_id]["kps"], (0, 230, 0), 5)
    if rec.get("detected"):
        skel(rec["detected"], (255, 30, 30), 3)
    txt = (f"{label} | {rec['verdict']} | PCK {rec['pck']:.2f} (L/R-swapped {rec['pck_swapped']:.2f})"
           + (f" | rules {rec['rules_pass']}/{rec['rules_total']}" if rec.get("rules_total") else "")
           + " | green=target red=detected")
    d.rectangle([0, 0, out.width, 26], fill=(255, 255, 255))
    d.text((6, 5), txt, fill=(0, 0, 0), font=_font(15))
    if rec["flags"]:
        d.rectangle([0, 26, out.width, 50], fill=(255, 236, 200))
        d.text((6, 31), " / ".join(rec["flags"])[:180], fill=(140, 40, 0), font=_font(14))
    return out

# 검증기 자체 점검: 모델이 정상적으로 불러와졌는지 스케치 한 장으로 돌려 봅니다.
_self = detect_people(render_sketch(POSES["pose_01"]["kps"]))
print(f"검증기 준비 완료. (스케치에서 사람 {len(_self)}명 검출 — 0명이어도 정상일 수 있음: 스케치는 사진이 아니므로)")

## 8. 생성 함수 — 경로 A(참조 이미지, ControlNet 아님) / 경로 B(SDXL ControlNet + Klein)

In [ ]:
# [이 셀이 하는 일] 이미지 생성 함수를 정의합니다. (아직 생성은 하지 않음)
#
# 경로 A — FLUX.2 Klein 4B "참조 이미지" 방식  ※ 정식 ControlNet이 아닙니다.
#   포즈 스케치(또는 OpenPose 뼈대)를 Klein의 image= 인자(멀티 레퍼런스 편집 입력)로 넣고,
#   "이 그림의 자세를 따르라"는 지시문을 프롬프트 앞에 붙입니다. 모델이 자세를 따를지는 보장되지 않고,
#   그래서 모든 결과를 7번 셀의 verify()로 검사해 기록합니다.
#
# 경로 B — SDXL + OpenPose ControlNet(정식) → Klein 4B 다듬기
#   1) SDXL ControlNet이 OpenPose 뼈대를 조건으로 초안을 그림 (자세 고정 담당)
#   2) Klein 4B가 초안을 참조 이미지로 받아 BASE-01 질감·인물·소품으로 다듬음 (화질 담당)
#   모델 교체가 느리므로 B 작업은 "초안 전부 → 다듬기 전부" 순서로 묶어서 돌립니다.
#
# 모든 결과는 이미지 + 검증 이미지(_verify.png) + 기록(results.json)으로 남습니다.
POSE_APPROVED = globals().get("POSE_APPROVED", False)

def _ref_image(pose_id, ref_kind, size=None):
    img = Image.open(SAMPLES / (f"{pose_id}.png" if ref_kind == "sketch" else f"{pose_id}_openpose.png")).convert("RGB")
    return img.resize(size, Image.LANCZOS) if size and img.size != tuple(size) else img

def _save_record(rec):
    rows = json.loads(RESULTS_JSON.read_text(encoding="utf-8")) if RESULTS_JSON.exists() else []
    rows = [r for r in rows if r["file"] != rec["file"]] + [rec]     # 같은 파일을 다시 만들면 기록을 덮어씀
    RESULTS_JSON.write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")

def _finish(img, job, out_path, ref_kind_for_leak, extra=None):
    img.save(out_path)
    rec = verify(img, job["pose"], ref_kind_for_leak)
    label = f"{job['exp']} {job['pose']} {job['prompt']} {job['path']} seed{job['seed']}"
    draw_verify(img, job["pose"], rec, label).save(out_path.with_name(out_path.stem + "_verify.png"))
    row = {"file": out_path.relative_to(REPO_DIR).as_posix(), **{k: job[k] for k in ("exp", "pose", "prompt", "seed", "path")},
           "control": ("NOT ControlNet: Klein reference image" if job["path"].startswith("A") else
                       "SDXL OpenPose ControlNet + Klein refine"),
           **{k: rec.get(k) for k in ("verdict", "pck", "pck_swapped", "mean_err_px", "rules_pass", "rules_total",
                                      "n_people", "flags", "per_joint", "leak")},
           **(extra or {})}
    _save_record(row)
    print(f"  {out_path.name}: {rec['verdict']} PCK={rec['pck']:.2f} "
          + (f"rules={rec['rules_pass']}/{rec['rules_total']} " if rec.get('rules_total') else "")
          + (" ⚠ " + " / ".join(rec["flags"]) if rec["flags"] else ""))
    return row

def _klein(embed_key, ref_img, seed):
    pipe = ensure_klein()
    g = torch.Generator(DEVICE).manual_seed(seed)
    with torch.no_grad():
        ref_img = ref_img.resize((GEN_W, GEN_H), Image.LANCZOS) if ref_img.size != (GEN_W, GEN_H) else ref_img
        return pipe(image=[ref_img], prompt=None, prompt_embeds=EMBEDS[embed_key].to(DEVICE),
                    height=GEN_H, width=GEN_W, num_inference_steps=KLEIN_STEPS, guidance_scale=1.0,
                    generator=g).images[0]

def _out_path(job):
    return DIRS[job["dir"]] / f"{job['pose']}__{job['prompt']}__{job['path']}__s{job['seed']}.png"

def run_jobs(jobs, b_conditioning_scale=0.75, b_steps=30):
    """jobs: dict(exp, dir, pose, prompt, seed, path) 목록. path ∈ {'A-sketch', 'A-openpose', 'B'}"""
    assert POSE_APPROVED, "3-5 셀에서 포즈를 확인한 뒤 POSE_APPROVED = True 로 바꾸고 실행하세요."
    rows = []
    for job in [j for j in jobs if j["path"].startswith("A")]:
        ref_kind = job["path"].split("-")[1]
        img = _klein((job["prompt"], f"A_{ref_kind}"), _ref_image(job["pose"], ref_kind), job["seed"])
        rows.append(_finish(img, job, _out_path(job), ref_kind))
    b_jobs = [j for j in jobs if j["path"] == "B"]
    if b_jobs:
        pipe = ensure_sdxl()
        drafts = []
        for job in b_jobs:      # 1단계: SDXL ControlNet 초안
            g = torch.Generator(DEVICE).manual_seed(job["seed"])
            draft = pipe(prompt=PROMPTS[job["prompt"]]["sdxl"], negative_prompt=SDXL_NEG,
                         image=_ref_image(job["pose"], "openpose"), controlnet_conditioning_scale=b_conditioning_scale,
                         num_inference_steps=b_steps, guidance_scale=6.0, height=HEIGHT, width=WIDTH,
                         generator=g).images[0]
            dpath = _out_path(job).with_name(_out_path(job).stem.replace("__B__", "__B-draft__") + ".png")
            drafts.append((job, draft, dpath))
            print("  초안:", dpath.name)
        # 중요: 지역 변수 pipe가 SDXL을 계속 붙잡고 있으면 unload 후에도 GPU 메모리가 안 풀려
        #       Klein 로드에서 CUDA OOM이 납니다(T4 실측). 다듬기 전에 참조를 지우고 SDXL을 내립니다.
        del pipe
        unload("sdxl")
        # 초안도 따로 검증해 둡니다 (ControlNet 단계에서 자세가 맞았는지 / 다듬기에서 무너졌는지 구분하기 위해)
        for job, draft, dpath in drafts:
            _finish(draft, {**job, "path": "B-draft"}, dpath, None, {"cn_scale": b_conditioning_scale})
        for job, draft, dpath in drafts:   # 2단계: Klein 다듬기
            img = _klein((job["prompt"], "B_refine"), draft, job["seed"])
            rows.append(_finish(img, job, _out_path(job), None, {"draft": dpath.relative_to(REPO_DIR).as_posix(),
                                                                   "cn_scale": b_conditioning_scale}))
    return rows

def make_grid(rows, cols, title, out_name, thumb=(448, 256)):
    """결과(검증 오버레이 포함)를 표로 모은 비교 이미지."""
    rows = list(rows)
    n_rows = math.ceil(len(rows) / cols)
    tw, th = thumb
    sheet = Image.new("RGB", (cols * tw, n_rows * (th + 34) + 34), (255, 255, 255))
    d = ImageDraw.Draw(sheet)
    d.text((6, 8), title, fill=(0, 0, 0), font=_font(18))
    for i, r in enumerate(rows):
        x, y = (i % cols) * tw, 34 + (i // cols) * (th + 34)
        im = Image.open(REPO_DIR / r["file"].replace(".png", "_verify.png")).resize(thumb)
        sheet.paste(im, (x, y + 34))
        d.text((x + 4, y + 2), f"{r['pose']} | {r['prompt']} | {r['path']} s{r['seed']}", fill=(0, 0, 0), font=_font(13))
        d.text((x + 4, y + 18), f"{r['verdict']}  PCK {r['pck']:.2f}", fill=(0, 0, 160), font=_font(13))
    sheet.save(SAMPLES / out_name)
    return sheet

print("생성 함수 준비 완료. POSE_APPROVED =", POSE_APPROVED)

## 9. BASE-01 실행 — 경로 A
스케치 참조와 뼈대 참조를 각각 시드 3개로 돌려 6장을 만듭니다. 가장 가까운 결과를 `output_01.png`로 저장하고,
**판정이 SUCCESS가 아니면 그 사실을 `output_01.json`에 그대로 남깁니다** (다른 자세로 바꿔 성공이라고 하지 않습니다).

In [ ]:
# [이 셀이 하는 일] BASE-01 자세로 경로 A를 실행합니다.
# - 참조 이미지 2종(스케치 / OpenPose 뼈대) × 시드 3개 = 6장을 만들고 모두 검증합니다.
# - 가장 점수가 높은 한 장을 samples/output_01.png 로 복사합니다. 판정이 SUCCESS가 아니어도
#   "가장 가까운 결과"로 저장하되, output_01.json 에 판정과 실패 사유를 그대로 적습니다(성공으로 바꿔 적지 않음).
import shutil
RANK = {"SUCCESS": 3, "PARTIAL": 2, "FAIL": 1, "NO_DETECTION": 0}

base_jobs = [dict(exp="base", dir="base", pose="pose_01", prompt="BASE_01", seed=s, path=p)
             for p in ("A-sketch", "A-openpose") for s in BASE_SEEDS]
base_rows = run_jobs(base_jobs)

def pick_best(rows):
    return max(rows, key=lambda r: (RANK[r["verdict"]], r["pck"], r.get("rules_pass") or 0))

def publish(row, out_id, pose_id):
    """선택된 결과를 과제 형식 이름(output_0X.png)으로 복사하고, 판정을 숨기지 않고 json으로 남깁니다."""
    src = REPO_DIR / row["file"]
    shutil.copy(src, SAMPLES / f"{out_id}.png")
    shutil.copy(str(src).replace(".png", "_verify.png"), SAMPLES / f"{out_id}_verify.png")
    (SAMPLES / f"{out_id}.json").write_text(json.dumps(
        {**row, "pose_reference": f"samples/{pose_id}.png",
         "note": ("경로 A는 정식 ControlNet이 아니라 Klein 4B 참조 이미지 방식입니다." if row["path"].startswith("A")
                  else "경로 B: SDXL OpenPose ControlNet 초안 + Klein 4B 다듬기")},
        ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"{out_id}.png ← {row['file']}  [{row['verdict']}, PCK {row['pck']:.2f}]")

best_A = pick_best(base_rows)
publish(best_A, "output_01", "pose_01")
make_grid(base_rows, 3, "BASE-01 / path A (Klein reference image, NOT ControlNet)", "base01_pathA_grid.png")
NEED_PATH_B = best_A["verdict"] != "SUCCESS"
print("\n경로 A 최고 판정:", best_A["verdict"], "→", "경로 B 실행 권장" if NEED_PATH_B else "경로 A로 충분")

## 10. 경로 B — 경로 A가 자세를 따르지 못할 때

| 항목 | 내용 |
|---|---|
| 포즈 조건 모델 | `xinsir/controlnet-openpose-sdxl-1.0` — SDXL 1.0 전용 `ControlNetModel`, **Apache-2.0**, 2.5GB |
| 기반 모델 | `stabilityai/stable-diffusion-xl-base-1.0` (fp16 변형) — **CreativeML Open RAIL++-M**, 6.9GB |
| 다듬기 모델 | `black-forest-labs/FLUX.2-klein-4B` — **Apache-2.0** (경로 A와 같은 모델) |
| 검증용 | `lllyasviel/Annotators` body_pose_model — CMU OpenPose, **비상업 용도만** |
| GPU | SDXL+ControlNet fp16 ≈ 9~10GB → T4(15GB) 가능. Klein과 동시에 올리지 않고 번갈아 올립니다 |
| 호환성 주의 | 이 ControlNet은 SDXL용이라 FLUX 계열(Klein)에 직접 붙일 수 없습니다. 그래서 "SDXL로 자세 고정 → Klein으로 질감" 2단계입니다 |
| 한계 | SDXL의 CLIP은 77토큰까지만 읽어 짧은 프롬프트를 씁니다. Klein 다듬기 단계에서 자세가 다시 흐트러질 수 있으므로 초안과 최종을 따로 검증합니다 |

먼저 메타데이터로 라이선스와 크기를 확인합니다 (다운로드 없음).

In [ ]:
# [이 셀이 하는 일] 경로 B를 돌리기 전에 모델·라이선스·GPU 자원을 확인합니다. (다운로드 없음, API로 메타데이터만 조회)
from huggingface_hub import HfApi
api = HfApi()
for rid, use in [(SDXL_ID, "기반 모델"), (SDXL_POSE_CN_ID, "OpenPose ControlNet")]:
    info = api.model_info(rid, files_metadata=True)
    lic = (info.card_data or {}).get("license") if info.card_data else None
    size = sum((s.size or 0) for s in info.siblings
               if (rid == SDXL_ID and s.rfilename.endswith("fp16.safetensors") and "/" in s.rfilename)
               or (rid == SDXL_POSE_CN_ID and s.rfilename in ("config.json", "diffusion_pytorch_model.safetensors")))
    print(f"{use:20s} {rid:42s} license={lic}  받을 크기≈{size / 1e9:.1f}GB  gated={info.gated}")
print(f"GPU {ENV['gpu']} {ENV['vram_gb']}GB | 디스크 여유 {shutil.disk_usage('.').free / 1e9:.0f}GB")
print("호환성: xinsir/controlnet-openpose-sdxl-1.0 은 SDXL 1.0 전용 ControlNetModel 입니다 (FLUX 계열에는 못 씀).")
print("SDXL fp16 + ControlNet ≈ GPU 9~10GB → T4(15GB)에서 단독으로 올릴 수 있습니다. Klein은 그동안 내립니다.")

### 10-1. 경로 B 실행 (기본: 경로 A가 SUCCESS가 아닐 때만)

In [ ]:
# [이 셀이 하는 일] 경로 B 실행: SDXL OpenPose ControlNet 초안 → Klein 4B 다듬기 (BASE-01, 시드 3개)
# - RUN_PATH_B = "auto" 이면 경로 A가 SUCCESS가 아닐 때만 실행합니다. True/False로 강제할 수 있습니다.
# - 초안(B-draft)과 다듬은 결과(B)를 따로 검증하므로, 자세가 어느 단계에서 무너졌는지 알 수 있습니다.
RUN_PATH_B = "auto"
B_CN_SCALE = 0.75     # ControlNet 세기 (모델 카드 권장 범위 근처). 자세를 더 강하게 고정하려면 0.9~1.0
pathB_rows = []
if RUN_PATH_B is True or (RUN_PATH_B == "auto" and NEED_PATH_B):
    pathB_rows = run_jobs([dict(exp="base", dir="pathB", pose="pose_01", prompt="BASE_01", seed=s, path="B")
                           for s in BASE_SEEDS], b_conditioning_scale=B_CN_SCALE)
    best_B = pick_best(pathB_rows)
    make_grid(pathB_rows, 3, "BASE-01 / path B (SDXL OpenPose ControlNet + Klein refine)", "base01_pathB_grid.png")
    print("경로 B 최고 판정:", best_B["verdict"], f"PCK {best_B['pck']:.2f}")
    if (RANK[best_B["verdict"]], best_B["pck"]) > (RANK[best_A["verdict"]], best_A["pck"]):
        publish(best_B, "output_01", "pose_01")
        print("→ output_01 을 경로 B 결과로 교체했습니다.")
else:
    print("경로 B 건너뜀 (RUN_PATH_B =", RUN_PATH_B, ", NEED_PATH_B =", NEED_PATH_B, ")")

## 11. 실험 경로 결정
BASE-01에서 자세를 더 잘 따른 경로로 실험 2·3을 돌립니다.

In [ ]:
# [이 셀이 하는 일] 실험 2·3에 쓸 경로를 정합니다.
# - 기본: BASE-01에서 더 잘 따른 경로. (A가 성공하면 A, A가 실패하고 B가 더 나으면 B)
# - 경로 A의 참조 종류도 BASE-01에서 점수가 높았던 쪽(스케치/뼈대)을 씁니다.
best_overall = json.loads((SAMPLES / "output_01.json").read_text(encoding="utf-8"))
EXP_PATH = best_overall["path"]     # "A-sketch" / "A-openpose" / "B"
print("실험 경로:", EXP_PATH, "| 정식 ControlNet 여부:", "예(SDXL)" if EXP_PATH == "B" else "아니오(Klein 참조 이미지)")

## 12. 실험 2 — 같은 포즈, 다른 프롬프트
포즈(pose_01)와 시드는 고정하고 프롬프트만 바꿉니다.

| 키 | 바꾼 것 | 보려는 것 |
|---|---|---|
| BASE_01 | (대조군) | 기준 |
| P2_1_SUBJECT | 인물·복장 (50대 남성, 회색 정장) | 인물이 바뀌어도 자세가 유지되는가 |
| P2_2_STYLE | 질감·조명 (현대 디지털, 백색 주광) | 화풍만 바꾸면 자세는 그대로인가 |
| P2_3_CONFLICT | 자세 문장을 "똑바로 서 있음"으로 | 글과 포즈 그림이 충돌하면 어느 쪽이 이기는가 |

In [ ]:
# [이 셀이 하는 일] 실험 2 — 같은 포즈(pose_01), 프롬프트만 변경. 시드도 같게 두어 프롬프트 효과만 봅니다.
#   BASE_01(대조군) / P2_1 인물·복장 변경 / P2_2 질감·조명 변경 / P2_3 자세와 충돌하는 문장
exp2_jobs = [dict(exp="exp2", dir="exp2", pose="pose_01", prompt=k, seed=s, path=EXP_PATH)
             for k in ("BASE_01", "P2_1_SUBJECT", "P2_2_STYLE", "P2_3_CONFLICT") for s in EXP_SEEDS]
exp2_rows = run_jobs(exp2_jobs, b_conditioning_scale=globals().get("B_CN_SCALE", 0.75))
make_grid(exp2_rows, len(EXP_SEEDS), f"EXP2 same pose (pose_01), different prompts | path {EXP_PATH}", "exp2_grid.png")

## 13. 실험 3 — 같은 프롬프트, 다른 포즈
프롬프트를 고정하고 포즈만 바꿉니다 (pose_01 복합 자세 / pose_02 선 채로 손 들기 / pose_03 벤치에 바르게 앉기).
BASE_01 원문(자세 묘사 포함)과 자세 문장을 뺀 BASE_01_NEUTRAL 두 가지로 돌려, 글의 자세 묘사가 포즈 그림을 방해하는지도 봅니다.

In [ ]:
# [이 셀이 하는 일] 실험 3 — 같은 프롬프트, 포즈만 변경 (pose_01 / pose_02 / pose_03).
#   프롬프트 두 가지로 돌립니다:
#   - BASE_01        : 원문에 pose_01 자세 묘사가 들어 있음 → 포즈 그림과 글이 충돌할 때 무엇이 이기는지
#   - BASE_01_NEUTRAL: 자세 문장을 뺀 판 → 순수하게 포즈 그림만 바뀌었을 때 따라오는지
exp3_jobs = [dict(exp="exp3", dir="exp3", pose=p, prompt=k, seed=s, path=EXP_PATH)
             for k in ("BASE_01", "BASE_01_NEUTRAL") for p in ("pose_01", "pose_02", "pose_03") for s in EXP_SEEDS]
exp3_rows = run_jobs(exp3_jobs, b_conditioning_scale=globals().get("B_CN_SCALE", 0.75))
make_grid(exp3_rows, len(EXP_SEEDS) * 3, f"EXP3 same prompt, different poses | path {EXP_PATH}", "exp3_grid.png")
# 과제 형식의 pose_02/output_02 (+ pose_03/output_03): 자세 중립 프롬프트 결과 중 가장 높은 것
for pid, oid in (("pose_02", "output_02"), ("pose_03", "output_03")):
    publish(pick_best([r for r in exp3_rows if r["pose"] == pid and r["prompt"] == "BASE_01_NEUTRAL"]), oid, pid)

## 14. 결과 요약 표

In [ ]:
# [이 셀이 하는 일] 모든 기록(results.json)을 표로 묶어 samples/results_summary.md 로 저장합니다.
# - 숫자만 정리합니다. 마지막 마크다운 셀의 관찰은 이 표와 _verify.png 를 직접 보고 사람이 적습니다.
rows = json.loads(RESULTS_JSON.read_text(encoding="utf-8"))
lines = ["| 실험 | 포즈 | 프롬프트 | 경로 | 시드 | 판정 | PCK | 좌우반전 PCK | BASE 규칙 | 경고 | 파일 |",
         "|---|---|---|---|---|---|---|---|---|---|---|"]
for r in sorted(rows, key=lambda r: (r["exp"], r["pose"], r["prompt"], r["path"], r["seed"])):
    rules = f"{r['rules_pass']}/{r['rules_total']}" if r.get("rules_total") else "-"
    lines.append(f"| {r['exp']} | {r['pose']} | {r['prompt']} | {r['path']} | {r['seed']} | {r['verdict']} | "
                 f"{r['pck']:.2f} | {r['pck_swapped']:.2f} | {rules} | {'; '.join(r['flags']) or '-'} | {r['file']} |")

def agg(filter_fn):
    sel = [r for r in rows if filter_fn(r)]
    return (sum(r["pck"] for r in sel) / len(sel), len(sel)) if sel else (float("nan"), 0)

summary = ["# 결과 요약 (자동 생성)", "",
           "경로 A = FLUX.2 Klein 4B 참조 이미지 방식 (**정식 ControlNet 아님**) / "
           "경로 B = SDXL OpenPose ControlNet 초안 + Klein 4B 다듬기", "", "## 조건별 평균 PCK", ""]
for title, fn in [
    ("BASE-01 경로 A-스케치", lambda r: r["exp"] == "base" and r["path"] == "A-sketch"),
    ("BASE-01 경로 A-뼈대", lambda r: r["exp"] == "base" and r["path"] == "A-openpose"),
    ("BASE-01 경로 B-초안(SDXL)", lambda r: r["exp"] == "base" and r["path"] == "B-draft"),
    ("BASE-01 경로 B-다듬기", lambda r: r["exp"] == "base" and r["path"] == "B"),
    *[(f"실험2 {k}", (lambda k: lambda r: r["exp"] == "exp2" and r["prompt"] == k and r["path"] != "B-draft")(k))
      for k in ("BASE_01", "P2_1_SUBJECT", "P2_2_STYLE", "P2_3_CONFLICT")],
    *[(f"실험3 {p} × {k}", (lambda p, k: lambda r: r["exp"] == "exp3" and r["pose"] == p and r["prompt"] == k
                                                    and r["path"] != "B-draft")(p, k))
      for k in ("BASE_01", "BASE_01_NEUTRAL") for p in ("pose_01", "pose_02", "pose_03")],
]:
    m, n = agg(fn)
    if n:
        summary.append(f"- {title}: 평균 PCK {m:.2f} (n={n})")
summary += ["", "## 전체 기록", ""] + lines
(SAMPLES / "results_summary.md").write_text("\n".join(summary), encoding="utf-8")
print("\n".join(summary[:40]))

## 15. 결과 내려받기

In [ ]:
# [이 셀이 하는 일] samples 폴더를 zip으로 묶어 내려받습니다. (Colab 런타임이 끊기면 파일이 사라지므로)
# 내려받은 뒤 GitHub 저장소의 samples/ 에 그대로 올리면 됩니다.
shutil.make_archive("pose_tool_samples", "zip", REPO_DIR, "samples")
try:
    from google.colab import files
    files.download("pose_tool_samples.zip")
except ImportError:
    print("Colab이 아니면 pose_tool_samples.zip 파일을 직접 가져가세요.")

## 16. 실험 관찰 기록 — 무엇을 바꿔 보았고, 어떻게 달라졌는가

> 아래는 **직접 실행한 뒤** `samples/results_summary.md`, 각 그리드(`exp2_grid.png`, `exp3_grid.png`)와 `_verify.png`를 보고 채웁니다.
> 숫자(PCK·규칙 통과)와 눈으로 본 차이를 함께 적고, 실패한 경우 **실패한 그대로** 원인을 적습니다.

**실행 환경**: (GPU 종류 / 날짜 / 한 장당 생성 시간)

### A. 기준 자세 재현 (BASE-01)
- 경로 A-스케치: 평균 PCK ___, 최고 판정 ___ — 관찰: (예: 어느 관절이 어긋났나, 스케치 색이 남았나)
- 경로 A-뼈대: 평균 PCK ___, 최고 판정 ___ — 관찰:
- 경로 B-초안(SDXL ControlNet): 평균 PCK ___ / 경로 B-다듬기(Klein): 평균 PCK ___ — 관찰: (다듬기에서 자세가 유지됐나)
- `output_01.png`는 경로 ___ 결과, 판정 ___. 목표 자세와의 실제 차이:
- **정식 ControlNet 여부**: 경로 A는 아님 / 경로 B는 SDXL 단계에서만 정식 ControlNet

### B. 실험 2 — 같은 포즈, 프롬프트 변경
- 인물 변경(P2_1): PCK ___ → 관찰:
- 질감·조명 변경(P2_2): PCK ___ → 관찰:
- 충돌 문장(P2_3): PCK ___ → 관찰: (포즈 그림과 글 중 어느 쪽을 따랐나)
- 정리: 프롬프트를 바꿨을 때 크게 달라진 것 / 그대로였던 것:

### C. 실험 3 — 같은 프롬프트, 포즈 변경
- pose_02(선 자세): BASE_01 PCK ___ / NEUTRAL PCK ___ → 관찰:
- pose_03(앉은 자세): BASE_01 PCK ___ / NEUTRAL PCK ___ → 관찰:
- 정리: 포즈를 바꿨을 때 잘 따라온 것 / 따라오지 않은 것, 글의 자세 묘사가 방해했는지:

### D. 결론과 한계
- 어느 쪽(포즈 vs 프롬프트)을 바꿨을 때 결과가 더 크게 달라졌나:
- 이 복합 자세가 끝내 재현되지 않은 부분과 원인 추정:
- 검증 방법의 한계(OpenPose가 기이한 자세·거울 반사를 잘못 읽는 경우 등):